# Blackfly Camera Capture

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2

from camera.wrapper_pyspin import Blackfly
from utils.make_hdr import make_hdr

## Initialize Camera

In [ ]:
cam = Blackfly(pixel_format="BayerRG16")
cam.set_auto_exposure()

img, _, _, exposure, gain = cam.get_next_image(return_metadata=True)
print(f'shape: {img.shape} | exposure: {exposure:.1f} us | gain: {gain:.2f} dB | fps: {cam.get_fps():.2f}')

## Live Preview

Press `q` to quit.

In [ ]:
while True:
    frame, _, _, _, _ = cam.get_next_image(return_metadata=True)
    if frame is None:
        continue
    cv2.imshow('Blackfly', (frame / 256).astype('uint8'))
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cv2.destroyAllWindows()

## Capture Single Frame

In [ ]:
frame, _, _, exposure, gain = cam.get_next_image(return_metadata=True)

print(f'shape: {frame.shape}, dtype: {frame.dtype}, min: {frame.min()}, max: {frame.max()}')
print(f'exposure: {exposure:.1f} us | gain: {gain:.2f} dB')

frame_color = cv2.cvtColor((frame / 256).astype(np.uint8), cv2.COLOR_BayerRG2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(frame_color)
plt.title(f'Blackfly — BayerRG16 debayered | exposure: {exposure/1000:.2f} ms')
plt.axis('off')
plt.show()

#### Save single frame (PNG + raw npy)

In [ ]:
os.makedirs('./data/single', exist_ok=True)
timestamp = f'{time.time():.0f}'

cv2.imwrite(f'./data/single/blackfly_{timestamp}.png', frame_color[..., ::-1])
np.save(f'./data/single/blackfly_{timestamp}_raw.npy', frame)
print(f'Saved to ./data/single/blackfly_{timestamp}.png')

## HDR Capture

In [ ]:
num_steps = 8
ratio = 16
base_exposure = cam.get_exposure()
min_exp = max(cam.min_exp_val, base_exposure / ratio)
max_exp = min(cam.max_exp_val, base_exposure * ratio)
exposures = [float(e) for e in np.geomspace(min_exp, max_exp, num_steps)]
print(f'HDR exposures (us): {[f"{e:.1f}" for e in exposures]}')

cam.set_auto_exposure(False)

frames = []
real_exposures = []

for exp in exposures:
    cam.set_exposure(exp, check=False)
    time.sleep(0.05)
    frame, _, _, cur_exp, _ = cam.get_next_image(return_metadata=True)
    frames.append(frame)
    real_exposures.append(cur_exp)
    print(f'  {cur_exp:.1f} us — shape: {frame.shape}')

cam.set_exposure(base_exposure, check=False)
cam.set_auto_exposure()

frames = np.stack(frames, axis=-1)       # (H, W, N)
real_exposures = np.array(real_exposures)
print(f'frames shape: {frames.shape}')

#### Save raw multi-exposure stack

In [ ]:
os.makedirs('./data/hdr', exist_ok=True)
timestamp = f'{time.time():.0f}'

np.savez(f'./data/hdr/blackfly_{timestamp}.npz', frames=frames, exposures=real_exposures)
print(f'Saved to ./data/hdr/blackfly_{timestamp}.npz')

## HDR Synthesis

In [ ]:
hdr, ldr = make_hdr(frames, real_exposures, method='linear_raw')

print(f'HDR radiance map: shape={hdr.shape}, dtype={hdr.dtype}, min={hdr.min():.4f}, max={hdr.max():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(ldr[..., ::-1])
axes[0].set_title('LDR (Reinhard tone-mapped)')
axes[0].axis('off')
axes[1].imshow(np.log1p(hdr.mean(axis=2)), cmap='gray')
axes[1].set_title('HDR radiance map (log scale)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

#### Visualize exposure stack

In [ ]:
fig, axes = plt.subplots(1, len(real_exposures), figsize=(3 * len(real_exposures), 4))
for i, ax in enumerate(axes):
    ax.imshow((frames[..., i] / 256).astype(np.uint8), cmap='gray')
    ax.set_title(f'{real_exposures[i]/1000:.1f} ms')
    ax.axis('off')
plt.tight_layout()
plt.show()

#### Save HDR results (LDR preview + HDR radiance map)

In [ ]:
cv2.imwrite(f'./data/hdr/blackfly_{timestamp}_ldr.png', ldr)
cv2.imwrite(f'./data/hdr/blackfly_{timestamp}_radiance.hdr', hdr)
print(f'Saved: blackfly_{timestamp}_ldr.png, blackfly_{timestamp}_radiance.hdr')

## Cleanup

In [ ]:
cam.stop()